# Making Multiple Markets

## Overview

By diversifying your assets and constructing a market-making book, you can achieve improved risk-adjusted returns through the effects of diversification. In this example, we will demonstrate how the statistics of your market-making portfolio change as you increase the number of assets for which you create markets.

To implement Grid Trading using the GLFT market-making model across multiple assets universally without needing to adjust parameters, a few modifications are required:

Order quantities vary between assets due to differences in price, trading volume, and liquidity in the order book. To backtest all at once, you need to normalize your order quantities and make adjustments accordingly.

In certain assets, market trades primarily take place at the best bid and offer levels. Since we only calculate our trading intensity when market trades match our quotes, you may not achieve adequate trading intensity to suit your trading intensity function in such cases. As a result, you'll need to explore alternative methods to determine your half spread and skew based on order arrival depths or you need to increase your reaction interval to get more deeper order arrival depth but it leads you to react delayed especially in a fast-moving market.

See how $adj_2$ is determined to normalize different order quantities.

<div class="alert alert-info">
    
**Note:** This example is for educational purposes only and demonstrates effective strategies for high-frequency market-making schemes. All backtests are based on a 0.005% rebate, the highest market maker rebate available on Binance Futures. See <a href="https://www.binance.com/en/support/announcement/binance-updates-usd%E2%93%A2-margined-futures-liquidity-provider-program-2024-06-03-fefc6aa25e0947e2bf745c1c56bea13e">Binance Upgrades USDⓢ-Margined Futures Liquidity Provider Program</a> for more details.
    
</div>

<div class="alert alert-info">

**Note:** The trading intensity in  `out` of `measure_trading_intensity` is incorrectly in half-tick units instead of tick units. Although this fix requires adjusting parameters in all related examples, the example is left unchanged to preserve existing results.

</div>

## Runnable Tardis Test

This notebook executes the corresponding experiment through the shared
`tutorial_reproduction` runner. It uses existing Tardis files only and
does not require a Tardis API key or download data.

Defaults:

- amdserver: `/home/molly/data/tardis/binance-futures`, `2025-08-01`
- Mac: `~/Documents/tardis`, `2025-01-01`
- Window: `300` seconds

Optional environment overrides:

- `HFTBACKTEST_TARDIS_ROOT`
- `HFTBACKTEST_MULTI_TARDIS_ROOT`
- `HFTBACKTEST_TARDIS_DATE`
- `HFTBACKTEST_NOTEBOOK_SECONDS`
- `HFTBACKTEST_NOTEBOOK_OUTPUT`

Active experiment: `Making Multiple Markets.ipynb` (`making_multiple_markets`).
This file is a generated copy; the original notebook under `examples/` remains unchanged.

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
search_roots = []
for base in (cwd, *cwd.parents):
    search_roots.extend((base, base / 'examples'))
examples_root = next(
    path for path in search_roots
    if (path / 'tutorial_reproduction').is_dir()
)
if str(examples_root) not in sys.path:
    sys.path.insert(0, str(examples_root))

from tutorial_reproduction.notebook_support import (
    context_dict,
    notebook_context,
    run_notebook_experiment,
)

In [ ]:
context = notebook_context('0804T005')
context_dict(context)

In [ ]:
manifest = run_notebook_experiment('making_multiple_markets', context)
manifest['result']

In [ ]:
assert manifest['result']['status'] != 'failed'
print('notebook:', manifest['notebook'])
print('status:', manifest['result']['status'])
print('output:', context.output_root)

## Original Tutorial Reference

The original tutorial narrative and code are retained below for comparison.
Original code cells are rendered as non-executing references so that
`Run All` remains reproducible with the configured Tardis dataset.

The order quantity is determined to be equivalent to a notional value of $100.

By utilizing multiprocessing, backtesting of multiple assets can be conducted simultaneously.

You can see the equity curve of individual assets and notice how combining multiple assets can lead to a smoother equity curve, thereby enhancing risk-adjusted returns.

This presents an equity curve based on the number of assets, which are altcoins excluding BTC and ETH.

## Impact of Order Latency

When applying amplified feed latency, you can observe a decrease in performance due to the effects of latency.

When actual historical order latency is applied, the performance may deteriorate further compared to when amplified feed latency is used.

Therefore, understanding your order latency is crucial to achieving more precise backtest results. This understanding underscores the importance of latency reduction for market makers or high-frequency traders. This is why crypto exchanges not only offer maker rebates but also provide low-latency infrastructure to eligible market makers.

## Simpler model

So far, we only cover $\xi>0$ case, but $\xi=0$ case would be more simple and appropriate in practice especially in cryptocurrencies.

Revisit the equations (4.6) and (4.7) in [Optimal market making](https://arxiv.org/abs/1605.01862) and explore how they can be applied to real-world scenarios.

The optimal bid quote depth, $\delta^{b*}_{approx}$, and ask quote depth, $\delta^{a*}_{approx}$, are derived from the fair price as follows in the case of $\xi=0$:

\begin{align}
\delta^{b*}_{approx}(q) = {1 \over k} + {{2q + \Delta} \over 2}\sqrt{{{\gamma \sigma^2 e} \over {2A\Delta k}}} \label{eq4.6}\tag{4.6} \\
\delta^{a*}_{approx}(q) = {1 \over k} - {{2q - \Delta} \over 2}\sqrt{{{\gamma \sigma^2 e} \over {2A\Delta k}}} \label{eq4.7}\tag{4.7}
\end{align}

Let's introduce $c_1$ and $c_2$ and define them by extracting the volatility 𝜎 from the square root as same as before:

\begin{align}
c_1 = {1 \over k} \\
c_2 = \sqrt{{{\gamma e} \over {2A\Delta k}}}
\end{align}

Now we can rewrite equations (4.6) and (4.7) as follows:

\begin{align}
\delta^{b*}_{approx}(q) = c_1 + {\Delta \over 2} \sigma c_2 + q \sigma c_2 \\
\delta^{a*}_{approx}(q) = c_1 + {\Delta \over 2} \sigma c_2 - q \sigma c_2
\end{align}

It's more concise and only need to adjust $\gamma$ and its effect is more straightforward.

You can observe a more straight line in the equity curve with higher $\gamma$, which induces greater skew. However, it also experiences more severe drawdowns in fast-moving markets. Additionally, because of the higher skew, profits are diminished as there's a greater tendency to close the position.

## A Case for More Assets

The more assets you make a market for, the better risk-adjusted return you achieve. This effect becomes dramatically evident.